In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage
from langchain.chat_models import init_chat_model

model = init_chat_model(model="groq:qwen/qwen3-32b")
model.invoke("hi")

c:\Users\mendh\Langchain-Langgraph\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


AIMessage(content='<think>\nOkay, the user sent "hi". I need to respond appropriately. Since it\'s a greeting, I should greet them back and offer help. Let me make sure to keep it friendly and open-ended. Maybe ask how I can assist them today. Keep it simple and conversational.\n</think>\n\nHi! How can I assist you today? 😊', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 9, 'total_tokens': 82, 'completion_time': 0.131199689, 'completion_tokens_details': None, 'prompt_time': 0.000252646, 'prompt_tokens_details': None, 'queue_time': 0.047584864, 'total_time': 0.131452335}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d61c8-e878-7572-9dde-a17b0a9a168f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 73, 'total_tokens': 82})

In [3]:
agent = create_agent(
    model = 'groq:llama-3.1-8b-instant',
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = 'groq:llama-3.1-8b-instant',
            trigger = ("messages",10),
            keep = ("messages",4)
        )
    ]
)

In [4]:
config = {"configurable":{"thread_id":"1"}}

questions = [
    "what is 2+2?",
    "what is 10*5?",
    "what is 1000/4",
    "what is 15-7",
    "what is 3*3+1",
    "what is 4/4*1+1-2"
]

for question in questions:
    print(question)
    response = agent.invoke({"messages":[HumanMessage(content=question)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response["messages"])}")

what is 2+2?
Messages: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='a38794e6-9397-41df-acb4-0a69485683dc'), AIMessage(content='2 + 2 = 4.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 42, 'total_tokens': 51, 'completion_time': 0.009253458, 'completion_tokens_details': None, 'prompt_time': 0.002763738, 'prompt_tokens_details': None, 'queue_time': 0.045806947, 'total_time': 0.012017196}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d61c8-ea52-75e3-a472-77a34916e72e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 9, 'total_tokens': 51})]}
Messages: 2
what is 10*5?
Messages: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='a38794e6-9397-41df-acb

### Human in the loop MiddleWare

In [5]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

def read_email_tool(email_id):
    """Function to read an email by its mail id"""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipent,subject,body):
    """Function to send an email"""
    return f"Email sent to {recipent} with subject '{subject} and body {body}'"

In [6]:
agent = create_agent(
    model='groq:llama-3.1-8b-instant',
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
        interrupt_on={
            "send_email_tool":{
                "allowed_decisions":["approve","edit","reject"]
            },
            "read_email_tool":False,
        }
    )]
)

In [7]:
config = {"configurable":{"thread_id":"test-approve"}}

result = agent.invoke(
    {"messages": [HumanMessage(content="send email to 'john@test.com' with subject 'Hello' and body 'How are you?'")]},
    config=config
)

result

{'messages': [HumanMessage(content="send email to 'john@test.com' with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='2b6aabae-96fb-4b0c-8c39-ed3d5b029ee9'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '87d1dtwry', 'function': {'arguments': '{"body":"How are you?","recipent":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 305, 'total_tokens': 339, 'completion_time': 0.038840485, 'completion_tokens_details': None, 'prompt_time': 0.018245141, 'prompt_tokens_details': None, 'queue_time': 0.045244579, 'total_time': 0.057085626}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d61c8-efc5-7c10-a6c0-29532937b7ad-0', tool_calls=[{'name': 'send_email_tool', 'args': {'bod

### Approve

In [8]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config=config
    )

print(f"Result: {result["messages"][-1].content}")

paused! Approving...
Result: 


In [9]:
result

{'messages': [HumanMessage(content="send email to 'john@test.com' with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='2b6aabae-96fb-4b0c-8c39-ed3d5b029ee9'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '87d1dtwry', 'function': {'arguments': '{"body":"How are you?","recipent":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 305, 'total_tokens': 339, 'completion_time': 0.038840485, 'completion_tokens_details': None, 'prompt_time': 0.018245141, 'prompt_tokens_details': None, 'queue_time': 0.045244579, 'total_time': 0.057085626}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d61c8-efc5-7c10-a6c0-29532937b7ad-0', tool_calls=[{'name': 'send_email_tool', 'args': {'bod

### Reject

In [10]:
config = {"configurable":{"thread_id":"test-reject"}}

result = agent.invoke(
    {"messages":[HumanMessage(content="Send email to Vishal@email.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)



In [11]:
result

{'messages': [HumanMessage(content="Send email to Vishal@email.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='20b77364-5404-42d7-a3d5-95aabcaf3608'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '0kmm4dwqq', 'function': {'arguments': '{"body":"How are you?","recipent":"Vishal@email.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 304, 'total_tokens': 340, 'completion_time': 0.060365303, 'completion_tokens_details': None, 'prompt_time': 0.025114563, 'prompt_tokens_details': None, 'queue_time': 0.046300017, 'total_time': 0.085479866}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d61c9-01f4-7ca2-8aee-0c8f2c31dba5-0', tool_calls=[{'name': 'send_email_tool', 'args': {

In [12]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"reject"}
                ]
            }
        ),
        config=config
    )

print(f"Result: {result["messages"][-1].content}")

paused! Approving...
Result: 


In [13]:
result

{'messages': [HumanMessage(content="Send email to Vishal@email.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='20b77364-5404-42d7-a3d5-95aabcaf3608'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '0kmm4dwqq', 'function': {'arguments': '{"body":"How are you?","recipent":"Vishal@email.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 304, 'total_tokens': 340, 'completion_time': 0.060365303, 'completion_tokens_details': None, 'prompt_time': 0.025114563, 'prompt_tokens_details': None, 'queue_time': 0.046300017, 'total_time': 0.085479866}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d61c9-01f4-7ca2-8aee-0c8f2c31dba5-0', tool_calls=[{'name': 'send_email_tool', 'args': {

In [14]:
config = {"configurable":{"thread_id":"test-edit"}}

result = agent.invoke(
    {"messages":[HumanMessage(content="Send email to wrong@gamil.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)



In [15]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"edit",
                     "edited_action":{
                         "name":"send_email_tool",
                         "args":{
                             "recipient":"correct@email.com",
                             "subject":"Corrected subject",
                             "body":"This was edited by  human before sending"
                         }
                     }}
                ]
            }
        ),
        config=config
    )

print(f"Result: {result["messages"][-1].content}")


paused! Approving...


BadRequestError: Error code: 400 - {'error': {'message': "tool call validation failed: parameters for tool send_email_tool did not match schema: errors: [missing properties: 'recipent']", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=send_email_tool>{"recipient": "wrong@gmail.com", "subject": "Hello", "body": "How are you?"}</function>'}}